## Data Cleaning Pipeline

In [ ]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

C:\Users\Brian Munene M\AppData\Local\Temp\ipykernel_23628\4075429923.py:8: DtypeWarning: Columns (6: promo_id) have mixed types. Specify dtype option on import or set low_memory=False.
  trans_items_df = pd.read_csv(r"..\..\data\transaction_items.csv") #data\transaction_items.csv


In [18]:
# Find the project root
PROJECT_ROOT = Path.cwd().resolve().parents[1]

# Make the project root available to Python
sys.path.insert(0, str(PROJECT_ROOT))

# Load dataset
trans_items_df = pd.read_csv(r"..\..\data\transaction_items.csv") #data\transaction_items.csv

C:\Users\Brian Munene M\AppData\Local\Temp\ipykernel_23628\2114964283.py:8: DtypeWarning: Columns (6: promo_id) have mixed types. Specify dtype option on import or set low_memory=False.
  trans_items_df = pd.read_csv(r"..\..\data\transaction_items.csv") #data\transaction_items.csv


In [17]:
from src.data_cleaning import inspect_dataset

inspect_dataset(trans_items_df)


1. FIRST 3 ROWS
  transaction_id  line_no product_id  quantity  unit_price_kes  discount_pct  \
0       T0000001        1      P0137       1.0           378.0             0   
1       T0000001        2      P0017       2.0           309.0             0   
2       T0000001        3      P0187       1.0            99.0             0   
3       T0000002        1      P0212       1.0           651.0             0   
4       T0000002        2      P0206       1.0           108.0             0   

  promo_id  line_total_kes  
0      NaN           378.0  
1      NaN           618.0  
2      NaN            99.0  
3      NaN           651.0  
4      NaN           108.0  

2. DATASET SHAPE
Number of rows: 1315864
Number of columns: 8

3. COLUMN NAMES
['transaction_id', 'line_no', 'product_id', 'quantity', 'unit_price_kes', 'discount_pct', 'promo_id', 'line_total_kes']

4. DUPLICATE ROWS
Number of duplicate rows: 3936

6. MISSING VALUES
transaction_id          0
line_no                 0
product

In [3]:
# 1. HANDLE MISSING QUANTITIES

# Record the original row count
rows_before = len(trans_items_df)

# Count missing quantities
missing_quantity_count = trans_items_df["quantity"].isna().sum()

# Remove rows with missing quantities
df_clean = trans_items_df.dropna(subset=["quantity"]).copy()

# Report the result
rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print("=" * 50)
print("MISSING QUANTITY CLEANING")
print("=" * 50)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Missing quantities:   {missing_quantity_count:,}")
print(f"Rows removed:         {rows_removed:,}")
print(f"Rows after cleaning:  {rows_after:,}")

print("\nRemaining missing quantities:")
print(df_clean["quantity"].isna().sum())


MISSING QUANTITY CLEANING
Rows before cleaning: 1,315,864
Missing quantities:   1,317
Rows removed:         1,317
Rows after cleaning:  1,314,547

Remaining missing quantities:
0


In [4]:
# 2. HANDLE DUPLICATES USING TRANSACTION ID AND LINE NO AS KEYS

df_clean =  df_clean.copy()

# Check duplicate transaction-line keys
key_columns = ["transaction_id", "line_no"]

rows_with_duplicates = df_clean.duplicated(subset=key_columns).sum()

# Remove the duplicates
df_clean = df_clean.drop_duplicates(subset= key_columns, keep= "first").copy()

rows_without_duplicates = df_clean.duplicated(subset=key_columns).sum()


print("Duplicate transaction-line rows:", rows_with_duplicates)
print("Duplicate transaction-line row remaining:", rows_without_duplicates)

Duplicate transaction-line rows: 3930
Duplicate transaction-line row remaining: 0


In [5]:
# 3. HANDLE NEGATIVE QUANTITIES

negatives = df_clean["quantity"] < 0

negative_qnt_before = negatives.sum()

# Convert negatives to positives
df_clean.loc[negatives, "quantity"] = (
    df_clean.loc[negatives, "quantity"].abs()
)

negative_qnt_after = (df_clean["quantity"] < 0).sum()

print(f"Negative quantities before: {negative_qnt_before}")
print(f"Negative quantities after: {negative_qnt_after}")

Negative quantities before: 2618
Negative quantities after: 0


In [6]:
inspect_dataset(df_clean)


1. FIRST 3 ROWS
  transaction_id  line_no product_id  quantity  unit_price_kes  discount_pct  \
0       T0000001        1      P0137       1.0           378.0             0   
1       T0000001        2      P0017       2.0           309.0             0   
2       T0000001        3      P0187       1.0            99.0             0   
3       T0000002        1      P0212       1.0           651.0             0   
4       T0000002        2      P0206       1.0           108.0             0   

  promo_id  line_total_kes  
0      NaN           378.0  
1      NaN           618.0  
2      NaN            99.0  
3      NaN           651.0  
4      NaN           108.0  

2. DATASET SHAPE
Number of rows: 1310617
Number of columns: 8

3. COLUMN NAMES
['transaction_id', 'line_no', 'product_id', 'quantity', 'unit_price_kes', 'discount_pct', 'promo_id', 'line_total_kes']

4. DUPLICATE ROWS
Number of duplicate rows: 0

6. MISSING VALUES
transaction_id          0
line_no                 0
product_id

In [7]:
# 3. Handle missing values in promotion column

promo_id_before = df_clean["promo_id"].isna().sum()

df_clean["promo_id"] = df_clean["promo_id"].fillna("NO PROMOTION")

promo_id_after = df_clean["promo_id"].isna().sum()

print(f"Promo_ids before: {promo_id_before}")
print(f"Promo_ids after: {promo_id_after}")

Promo_ids before: 1253299
Promo_ids after: 0


In [11]:
print("=" * 70)
print("LARGEST LINE TOTAL")
print("=" * 70)

print(
    df_clean.nlargest(15, "line_total_kes")[
        ["transaction_id", "line_no", "product_id",
         "quantity", "unit_price_kes",
         "discount_pct", "promo_id", "line_total_kes"]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("LARGEST QUANTITIES")
print("=" * 70)

print(
    df_clean.nlargest(15, "quantity")[
        ["transaction_id", "line_no", "product_id",
         "quantity", "discount_pct",  "unit_price_kes", "line_total_kes"]
    ].to_string(index=False)
)

print("\n" + "=" * 70)
print("LARGEST UNIT PRICES")
print("=" * 70)

print(
    df_clean.nlargest(15, "unit_price_kes")[
        ["transaction_id", "line_no", "product_id",
         "quantity", "discount_pct", "unit_price_kes", "line_total_kes"]
    ].to_string(index=False)
)

LARGEST LINE TOTAL
transaction_id  line_no product_id  quantity  unit_price_kes  discount_pct     promo_id  line_total_kes
      T0309217        3      P0230       7.0          1563.0             0 NO PROMOTION         10941.0
      T0152842        3      P0231       5.0          1940.0             0 NO PROMOTION          9700.0
      T0306511        3      P0135       6.0          1591.0             0 NO PROMOTION          9546.0
      T0244237        1      P0135       6.0          1560.0             0 NO PROMOTION          9360.0
      T0023736        2      P0231       5.0          1854.0             0 NO PROMOTION          9270.0
      T0226498        1      P0135       6.0          1545.0             0 NO PROMOTION          9270.0
      T0294904        6      P0123       6.0          1495.0             0 NO PROMOTION          8970.0
      T0140991        9      P0230       6.0          1486.0             0 NO PROMOTION          8916.0
      T0065094        8      P0230       6.0 

In [9]:

# Correct only confirmed factor-of-100 unit-price errors

discount_factor = 1 - df_clean["discount_pct"] / 100

valid_rows = (
    df_clean["quantity"].gt(0)
    & df_clean["line_total_kes"].gt(0)
    & df_clean["unit_price_kes"].gt(0)
    & discount_factor.gt(0)
)

# Calculate the unit price implied by the recorded line total
implied_unit_price = (
    df_clean["line_total_kes"]
    / (df_clean["quantity"] * discount_factor)
)

# Identify prices recorded at approximately 100 times
# the price implied by the line total
price_ratio = df_clean["unit_price_kes"] / implied_unit_price

factor_100_mask = (
    valid_rows
    & price_ratio.between(99.99, 100.01)
)

rows_corrected = int(factor_100_mask.sum())

# Correct only the identified records
df_clean.loc[factor_100_mask, "unit_price_kes"] /= 100

print(f"Prices corrected: {rows_corrected:,}")

# Validate the corrected records
corrected = df_clean.loc[factor_100_mask]

expected_total = (
    corrected["quantity"]
    * corrected["unit_price_kes"]
    * (1 - corrected["discount_pct"] / 100)
)

difference = (
    corrected["line_total_kes"] - expected_total
).abs()

print(
    "Corrected records matching line totals:",
    int(difference.le(0.02).sum())
)
print(
    "Corrected records still needing investigation:",
    int(difference.gt(0.02).sum())
)


Prices corrected: 654
Corrected records matching line totals: 654
Corrected records still needing investigation: 0


In [12]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    output_dir / "transaction_items_clean.csv",
    index=False
)